<a href="https://colab.research.google.com/github/KMKomer/Sentiment-analysis/blob/main/sentement_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import nltk

In [95]:
df = pd.read_csv('drugLibTrain_raw.csv')
df.dropna()

le = LabelEncoder()

X_train, X_temp, Y_train, Y_temp = train_test_split(df['benefitsReview'],df['condition'], random_state =42, test_size = 0.3)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp,Y_temp, random_state =42, test_size = 0.5)

Y_train =le.fit_transform(Y_train)
Y_val = le.fit_transform(Y_val)
Y_test = le.fit_transform(Y_test)

device = torch.device('cpu')

tokenizer = AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment')


In [49]:
class reviewdataset(Dataset):
    def __init__(self, reviews, labels, tokenizer, maxlen=64):
        self.reviews = reviews
        self.labels = labels
        self.tokenizer = tokenizer
        self.maxlen = maxlen

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        label = self.labels[idx]


        tokens = self.tokenizer.encode_plus(review, add_special_tokens=True,return_token_type_ids=False,padding='max_length',max_length=self.maxlen, truncation =True,return_attention_mask=True, return_tensors ='pt')

        dic ={
            'input_ids': tokens['input_ids'].flatten(),
            'attention_mask':tokens['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

        return dic



In [50]:
train_dataset = reviewdataset(reviews=X_train.to_numpy(), labels = Y_train, tokenizer= tokenizer)
val_dataset = reviewdataset(reviews=X_val.to_numpy(), labels=Y_val, tokenizer=tokenizer)
test_dataset = reviewdataset(reviews=X_test.to_numpy(), labels=Y_test, tokenizer=tokenizer, maxlen=128)

train_loader = DataLoader(train_dataset, batch_size =16,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 16)
test_loader = DataLoader(test_dataset, batch_size = 16)
print(len(val_loader))


5


In [42]:
class classifier(nn.Module):
    def __init__(self, n_classes):
        super(classifier, self).__init__()
        self.bert = AutoModel.from_pretrained('cardiff/twitter-roberta-base-seniment')
        self.drop = nn.Dropout(p=0.3)
        self.linear1 = nn.Linear(768, n_classes)
        #self.linear2 = nn.Linear(512, 256)
        #self.linear3 = nn.Linear(256, 3)
    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last = output[0]
        output = last[:,0,:]
        print(output)
        out = self.drop(output)
        out = self.linear1(out)
        #out = self.linear2(out)
        #out = self.linear3(out)
        return out


In [135]:
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
huggingface_api_tokken = 'API token'
os.environ['HUGGINGFACE_TOKEN'] = huggingface_api_tokken
model = AutoModelForSequenceClassification.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment')
classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)


texts = []
scores = []
X = df['benefitsReview'].to_numpy()


X = [str(i) for i in X]

for i in X:

     texts.append(i)
     result = classifier(i)

     score = result[0]['score']
     score = np.round(score, decimals=2)

     scores.append(score)


results = pd.DataFrame()

labels = []
for i in scores:
     if i > 0.6:
      label = 'Positive'
     elif i <= 0.6 and i >= 0.4:
      label = 'Neutral'
     else:
      label = 'Negative'

     labels.append(label)
results['Drug Name'] = df['urlDrugName']
results['Condition'] = df['condition']
results['Effectiveness'] = df['effectiveness']
results['Review'] = texts
results['Score'] = scores
results['label'] = labels
results['Rating'] = df['rating']

results.head(5)
model_results = results.to_csv('model_results.csv')



Device set to use cpu


In [80]:
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
huggingface_api_tokken = 'API token'
os.environ['HUGGINGFACE_TOKEN'] = huggingface_api_tokken
model = AutoModelForSequenceClassification.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment')
classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)
#x = []
# The loop below caused KeyError: 0 due to direct integer indexing on a pandas Series
for i in range(len(X_train)):
     None
     #text = X_train.iloc[i]
     #result = classifier(X_train.iloc[i])
     #x.append(result)
     #print(f'Input:text, Output:{result}')
print(len(x), len(X_train))
# The previous error occurred because the pipeline expects raw text, not pre-tokenized data from the dataset.
# We will pass the raw text from X_train directly to the classifier after handling potential NaN values.
#all_predictions = classifier(X_train.fillna('').to_list()) # Convert Series to list of strings, filling NaNs

#print(f"Total predictions collected: {len(all_predictions)}")
#if len(all_predictions) > 0:
    #print(f"First 5 collected predictions: {all_predictions[:5]}")

Device set to use cpu


36 2174


In [44]:
optimizer = AdamW(model.parameters(), lr=3e-5, eps =1e-8)
loss_fn = nn.CrossEntropyLoss().to(device)
epouchs = 10
total_steps = len(train_loader) * epouchs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps = total_steps
)


def train(model,train_loader, optimizer,scheduler, device, loss_fn, n_examples):
    model.train()
    losses =[]
    corrected_preds = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)
        _, preds= torch.max(outputs, dim=1)

        loss = loss_fn(outputs, labels)
        losses.append(loss.item())

        corrected_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),max_norm = 1.0)

        optimizer.step()
        scheduler.step()

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    # Calculate metrics for the entire epoch
    epoch_accuracy = corrected_preds / total_samples
    avg_epoch_loss = np.mean(losses)

    return epoch_accuracy, avg_epoch_loss


In [45]:
def evaluate(model, val_loader, device, loss_fn, n_examples):
    print('\nEvaluating...')
    model.eval()
    corrected_preds = 0
    losses = []
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
         for batch_idx, batch_data in enumerate(val_loader):
                input_ids = batch_data['input_ids'].to(device)
                mask = batch_data['attention_mask'].to(device)
                labels = batch_data['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=mask)
                _,preds = torch.max(outputs, dim=1)
                loss = loss_fn(outputs, labels)
                losses.append(loss.item())

                corrected_preds += torch.sum(preds == labels).item()
                total_samples += labels.size(0)

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

    # Calculate metrics for the entire evaluation
    epoch_accuracy = corrected_preds / total_samples
    avg_epoch_loss = np.mean(losses)

    return epoch_accuracy, avg_epoch_loss


In [46]:
%%time

best_accuracy = 0

for epouch in range(epouchs):
    print(f'Epouch {epouch+1}/{epouchs}')
    print('='*70)
    train_acc , mean_loss = train(model, train_loader, optimizer, scheduler, device, loss_fn, n_examples = len(X_train))
    print(f'Train accuracy : {train_acc:.4f}')
    print(f'Mean loss :{mean_loss:.4f}')
    # Removed verbose print of predictions vs labels for training epochs

    print('='*70)
    val_accuracy , mean = evaluate(model, val_loader, device, loss_fn, n_examples = len(X_val))
    print(f'Validation accuracy : {val_accuracy:.4f}')
    print(f'Mean loss : {mean:.4f}')



    #if val_accuracy > best_accuracy:
        #best_accuracy = val_accuracy

        #print(f'best accuracy =  {val_accuracy:.4f}')
    #print('='*70)
#torch.save(model.state_dict(), 'sentimentanalyst.bin')

Epouch 1/10
tensor([[-0.2124,  0.0090,  0.0362,  ..., -0.6831,  0.4964,  0.1551],
        [ 0.1305,  0.3476,  0.3910,  ..., -0.2767,  0.5354,  0.6518],
        [ 0.1826,  0.1479,  0.1860,  ..., -0.1793,  0.0549,  0.3741],
        ...,
        [ 0.1271,  0.2092,  0.3496,  ..., -0.1463,  0.0892,  0.2126],
        [ 0.6178,  0.1900,  0.5701,  ..., -0.1219, -0.0104,  0.3552],
        [-0.0276, -0.2926, -0.1922,  ..., -0.5331,  0.0113, -0.1380]],
       grad_fn=<SelectBackward0>)
tensor([[-0.0795,  0.3826,  0.0811,  ..., -0.2072,  0.1454,  0.2619],
        [-0.0108,  0.7605, -0.1169,  ..., -0.4786,  0.6763,  0.4711],
        [-0.3763, -0.0995,  0.3859,  ..., -0.2085,  0.5126, -0.0460],
        ...,
        [ 0.2684,  0.4526,  0.7016,  ..., -0.0547,  0.1788,  0.5878],
        [-0.0614,  0.0717,  0.0557,  ..., -0.4302,  0.3350,  0.6100],
        [-0.1359,  0.1296, -0.5845,  ..., -0.0981,  0.1090,  0.4223]],
       grad_fn=<SelectBackward0>)
tensor([[-0.0069,  0.5921,  0.1209,  ..., -0.4055,  

KeyboardInterrupt: 

In [ ]:
print('===============Testing The Model==================')
test_accuracy, mean = evaluate(model, test_loader, device, loss_fn, n_examples = len(X_test))
#print('Predictions vs Labels', test_preds, test_labels )
print(f'Testing accuracy : {test_accuracy:.4f}')
print(f'Mean loss : {mean:.4f}')

===============Testing The Model==================

Evaluating...
Testing accuracy : 0.6800
Mean loss : 1.9955


In [128]:
x = [1, 5, 5, 10 ]
n = []
for i in x:
   if i == 1:
      l = 'p'
   elif i == 5:
      l = 'l'
   else:
      l = 'm'
   n.append(l)
print(n)

['p', 'l', 'l', 'm']


In [131]:
j = 0.34672898
print(f'{np.round(j, decimals=2)}')

0.35
